In [ ]:
# ===============================
# CELL 0 — Install (run once)
# ===============================
!pip install -q pandas numpy pyarrow fastparquet spacy tqdm
#!python -m spacy download en_core_web_sm -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 82.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 154.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ===============================
# CELL 1 — Mount & Paths
# ===============================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, html, unicodedata, hashlib
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/PropInsight"

# --- Input: your CSV in reddit_common ---
INPUT_FILE_DRIVE = f"{BASE}/preprocess/reddit_common/reddit_corpus_2023_2025.csv"
INPUT_FILE_LOCAL = "/mnt/data/reddit_corpus_2023_2025.csv"  # fallback if uploaded here in this chat
INPUT_FILE = INPUT_FILE_DRIVE if os.path.exists(INPUT_FILE_DRIVE) else INPUT_FILE_LOCAL

# --- Domain corpora (optional enrichers) ---
SINGLEX_CSV  = f"{BASE}/corpus/Singlish/lexicon.csv"
REGEX_JSONL  = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"
RULER_ORIG   = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
VOCAB_DIR    = f"{BASE}/corpus/SGPropertyDomain/vocab"
RULER_MERGED = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl"

# --- Outputs: same folder as input ---
OUTDIR       = f"{BASE}/labeled/reddit_common"
HASH_CHECKPT = f"{BASE}/labeled/checkpoints/reddit_common/reddit_hashes.parquet"
Path(OUTDIR).mkdir(parents=True, exist_ok=True)
Path(Path(HASH_CHECKPT).parent).mkdir(parents=True, exist_ok=True)

print("INPUT_FILE:", INPUT_FILE)
print("OUTDIR:", OUTDIR)
print("HASH_CHECKPT:", HASH_CHECKPT)


Mounted at /content/drive
INPUT_FILE: /content/drive/MyDrive/PropInsight/preprocess/reddit_common/reddit_corpus_2023_2025.csv
OUTDIR: /content/drive/MyDrive/PropInsight/labeled/reddit_common
HASH_CHECKPT: /content/drive/MyDrive/PropInsight/labeled/checkpoints/reddit_common/reddit_hashes.parquet


In [ ]:
# ===============================
# CELL 2 — Helpers
# ===============================
import glob
from tqdm.auto import tqdm

def normalize_ws(s: str) -> str:
    import re
    return re.sub(r"\s+", " ", s or "").strip()

def clean_basic(text: str) -> str:
    import re, html, unicodedata
    if not isinstance(text, str): return ""
    s = html.unescape(text)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\[.*?\]\(https?://[^\s)]+\)", " ", s)
    s = re.sub(r"https?://\S+|www\.\S+|\S+@\S+\.\S+", " ", s)
    s = re.sub(r"^\s*\[(deleted|removed)\]\s*$", " ", s, flags=re.I)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"([.,!?;:]){2,}", r"\1", s)
    return normalize_ws(s)

def compile_regexes_from_jsonl(path: Path):
    import json, re
    pats=[]
    if not path.exists(): return pats
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if not ln: continue
            try:
                d=json.loads(ln); pat=d.get("pattern"); name=d.get("name","pattern")
                if pat: pats.append((f"rx_{name}", re.compile(pat, re.I)))
            except: pass
    return pats

def build_singlish_dict(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column missing in {csv_path}. Found: {df.columns.tolist()}")
    return set(df["word"].dropna().astype(str).str.strip().str.lower())

def find_singlish_terms(text: str, words_set):
    import re
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+", text or "")
    return sorted(set([t.lower() for t in toks if t.lower() in words_set]))


In [ ]:
# ===============================
# CELL 3 — Merge vocab → EntityRuler
# ===============================
import re, json
def merge_vocab_to_entityruler(vocab_dir: str, existing_jsonl: str, merged_out: str):
    p_voc, p_ex, p_out = Path(vocab_dir), Path(existing_jsonl), Path(merged_out)
    existing=[]
    if p_ex.exists():
        for ln in p_ex.read_text(encoding="utf-8", errors="ignore").splitlines():
            ln=ln.strip()
            if ln:
                try: existing.append(json.loads(ln))
                except: pass

    def phrase_to_token_pattern(phrase: str):
        phrase = re.sub(r"\s+", " ", phrase).strip()
        if not phrase: return None
        return [{"LOWER": t.lower()} for t in phrase.split(" ") if t]

    def lowers_from_pattern(pat):
        if isinstance(pat, str): return tuple(re.sub(r"\s+"," ",pat).lower().split(" "))
        if isinstance(pat, dict): return (str(pat.get("LOWER", pat.get("TEXT",""))).lower(),)
        if isinstance(pat, list):
            outs=[]
            for tok in pat:
                if isinstance(tok, dict): outs.append(str(tok.get("LOWER", tok.get("TEXT",""))).lower())
                else: outs.append(str(tok).lower())
            return tuple(outs)
        return (str(pat).lower(),)

    def pat_key(rec): return (rec.get("label",""), lowers_from_pattern(rec.get("pattern","")))

    merged=[]; seen=set()
    for rec in existing:
        k = pat_key(rec)
        if k not in seen: seen.add(k); merged.append(rec)

    if p_voc.exists():
        for txt in sorted(p_voc.glob("*.txt")):
            label = re.sub(r"[^A-Za-z0-9]+","_", txt.stem).strip("_").upper() or "DOMAIN"
            for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
                term = raw.strip()
                if not term: continue
                pat = phrase_to_token_pattern(term)
                if not pat: continue
                rec={"label":label,"pattern":pat,"id":term}
                k=pat_key(rec)
                if k not in seen: seen.add(k); merged.append(rec)

    p_out.parent.mkdir(parents=True, exist_ok=True)
    with p_out.open("w", encoding="utf-8") as f:
        for rec in merged:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[EntityRuler] merged → {p_out} (rules: {len(merged)})")
    return str(p_out)

ENTITYRULER_MERGED = merge_vocab_to_entityruler(VOCAB_DIR, RULER_ORIG, RULER_MERGED)


[EntityRuler] merged → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl (rules: 2158)
